# P8 · Prompt Grader & Rewriter: train BERT and produce all results
**Ayush Pisal · Roll No. 08**

1. **Runtime → Change runtime type → T4 GPU**
2. Put your GitHub repo URL in the first code cell. You can also upload the project zip instead.
3. **Runtime → Run all**. It takes about 10–15 min in total.

In [ ]:
REPO_URL = "https://github.com/ayushpisal007-wq/promptsmith.git"  # <- change this
HF_USER  = "ayushpisal"                                    # <- change this

import os
if not os.path.exists('promptsmith'):
    if False:
        # no repo yet: upload p8-prompt-grader.zip instead
        from google.colab import files
        up = files.upload()
        !unzip -q -o {list(up)[0]}
    else:
        !git clone -q $REPO_URL
%cd promptsmith
!ls

In [ ]:
!pip install -q transformers==5.23.0 diffusers==0.43.0 accelerate tabulate
!nvidia-smi --query-gpu=name,memory.total --format=csv

## U3: fine-tune BERT on the synthetic rubric-labelled prompts

In [ ]:
!python train_bert.py --epochs 3

In [ ]:
from IPython.display import Image, display
display(Image('results/bert_confusion_matrices.png'))

## U1–U5: all measured results
This cell runs the zero-shot and few-shot judge on the 30 gold prompts, the 5 weak prompts before/after (text outputs, sd-turbo images and CLIP scores), the safety-detector test and the counterfactual bias test.

In [ ]:
!python evaluate.py

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/summary.md').read()))
for w in ['W4', 'W5']:
    for tag in ['original', 'rewritten']:
        p = f'results/outputs/{w}_{tag}.png'
        if os.path.exists(p):
            print(w, tag); display(Image(p, width=256))

## Publish the BERT model so the Hugging Face Space can load it
Paste a **write** token from https://huggingface.co/settings/tokens when asked. The token stays in this Colab session only.

In [ ]:
from huggingface_hub import notebook_login, HfApi
notebook_login()

In [ ]:
repo = f'{HF_USER}/p8-bert-prompt-quality'
api = HfApi()
api.create_repo(repo, exist_ok=True)
api.upload_folder(folder_path='models/bert-prompt-quality', repo_id=repo)
print('Model at https://huggingface.co/' + repo, '\nSet BERT_MODEL =', repo, 'in your Space variables')

## Download the results for the report

In [ ]:
!zip -q -r p8_results.zip results
from google.colab import files
files.download('p8_results.zip')